# Testes do pipeline de Agenda

Notebook didático e objetivo para testar as funcionalidades da pasta `src/agenda`.

Fluxo coberto:
- Pré-processamento de arquivos `.txt` da agenda política;
- Geração de embeddings semânticos por chunks;
- Modelagem de tópicos (LDA) a partir dos tokens da agenda;
- Inspeção rápida de resultados.

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Descobre a raiz do projeto (pasta que contém ./src)
# Isso evita problemas de import dependendo de onde o notebook foi aberto.
# ------------------------------------------------------------------
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists():
            return candidate
    raise RuntimeError("Não foi possível localizar a raiz do projeto (pasta com ./src).")

PROJECT_ROOT = find_project_root(Path.cwd())
AGENDA_SRC = PROJECT_ROOT / "src" / "agenda"

# Adiciona src/agenda no path para importar os módulos locais.
if str(AGENDA_SRC) not in sys.path:
    sys.path.append(str(AGENDA_SRC))

from pre_processing import processar_todos_elementos
from embeddings import generate_agenda_embeddings_from_txt
from topics import load_party_tokens_dataframe, topics_main

print("PROJECT_ROOT:", PROJECT_ROOT)
print("AGENDA_SRC:", AGENDA_SRC)

/home/gpcmoura/Documents/Code/Msc/Repositories/political-discourse-analysis/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


PROJECT_ROOT: /home/gpcmoura/Documents/Code/Msc/Repositories/political-discourse-analysis
AGENDA_SRC: /home/gpcmoura/Documents/Code/Msc/Repositories/political-discourse-analysis/src/agenda


## 1) Pré-processamento da agenda

A célula abaixo processa os `.txt` de um elemento específico (modo teste) ou de todos os elementos.

- Para teste rápido, use `elemento_teste = "UNIAO"` (ou outro elemento existente).
- Para processar tudo, use `elemento_teste = None`.

In [2]:
# Caminhos de entrada e saída do pré-processamento
pasta_agenda_politica = PROJECT_ROOT / "data" / "party_agenda" / "party"
pasta_saida_tokens = PROJECT_ROOT / "data" / "party_agenda" / "preprocessing" / "tokenization" / "tokens"
pasta_saida_tokens.mkdir(parents=True, exist_ok=True)

# Ajuste para teste rápido
elemento_teste = "UNIAO"  # ex.: "UNIAO" | use None para processar todos

# Sempre reprocessa e sobrescreve os arquivos existentes
processar_todos_elementos(
    pasta_agenda_politica=pasta_agenda_politica,
    pasta_saida_tokens=pasta_saida_tokens,
    elemento_teste=elemento_teste,
)

/home/gpcmoura/Documents/Code/Msc/Repositories/political-discourse-analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9688.08it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[Elemento] UNIAO
  - Manifesto_Uniao_BRASIL.layout_aware.txt -> Manifesto_Uniao_BRASIL.layout_aware_tokens.txt

Concluído: 1 arquivo(s) processado(s) para o elemento UNIAO.


In [3]:
# Inspeção simples dos arquivos gerados
elemento_para_inspecao = elemento_teste if elemento_teste else "UNIAO"
base_saida = pasta_saida_tokens / elemento_para_inspecao

txt_files = sorted((base_saida / "TXT").glob("*.txt")) if (base_saida / "TXT").exists() else []
csv_files = sorted((base_saida / "CSV").glob("*.csv")) if (base_saida / "CSV").exists() else []

print("Pasta de inspeção:", base_saida)
print("Qtd TXT de tokens:", len(txt_files))
print("Qtd CSV preprocessados:", len(csv_files))

if csv_files:
    display(pd.read_csv(csv_files[0]).head(5))
else:
    print("Nenhum CSV encontrado para inspeção.")

Pasta de inspeção: /home/gpcmoura/Documents/Code/Msc/Repositories/political-discourse-analysis/data/party_agenda/preprocessing/tokenization/tokens/UNIAO
Qtd TXT de tokens: 1
Qtd CSV preprocessados: 1


,chunk_id,chunk_text,preprocess_agenda,tokens
0,0,Nossos Valores e Princípios O Brasil é reconhe...,valor princípio brasil reconhecer celebrar plu...,"[""valor"", ""princípio"", ""brasil"", ""reconhecer"",..."
1,1,"São muitas e diferentes as nossas origens, as ...",diferente origem cultura tradição costume valo...,"[""diferente"", ""origem"", ""cultura"", ""tradição"",..."
2,2,É ele que nunca deixa de pulsar em nossos cora...,nunca deixar pulsar coração fazer reagir pessi...,"[""nunca"", ""deixar"", ""pulsar"", ""coração"", ""faze..."
3,3,Esse Amor pelo Brasil está assentado na nossa ...,amor brasil assentar história comum luta compa...,"[""amor"", ""brasil"", ""assentar"", ""história"", ""co..."
4,4,Mas não é uma utopia.,utopia,"[""utopia""]"


## 2) Embeddings da agenda por chunks semânticos

A célula abaixo testa a função `generate_agenda_embeddings_from_txt` com um `.txt` da agenda.

Arquivo sugerido para teste:
`data/party_agenda/party/UNIAO/txt/Manifesto_Uniao_BRASIL.layout_aware.txt`

Saída esperada dos embeddings:
`data/party_agenda/embeddings/<PARTIDO>/`

In [4]:
# Ajuste aqui o arquivo de entrada
txt_path = PROJECT_ROOT / "data" / "party_agenda" / "party" / "UNIAO" / "txt" / "Manifesto_Uniao_BRASIL.layout_aware.txt"

# Define o partido com base no caminho do arquivo (..../party/<PARTIDO>/txt/arquivo.txt)
partido_atual = txt_path.parent.parent.name

# Pasta de saída dos embeddings por partido
output_dir = PROJECT_ROOT / "data" / "party_agenda" / "embeddings" / partido_atual
output_dir.mkdir(parents=True, exist_ok=True)

# Controle de reprocessamento
force_recompute_embeddings = False

# Mesmo padrão de nome usado em embeddings.py
safe_source_id = re.sub(r"[^\w\-.]", "_", txt_path.name)
pattern_csv = f"agenda_embeddings_{safe_source_id}_*.csv"
pattern_npy = f"agenda_embeddings_{safe_source_id}_*.npy"

existing_csv = sorted(output_dir.glob(pattern_csv), key=lambda p: p.stat().st_mtime)
existing_npy = sorted(output_dir.glob(pattern_npy), key=lambda p: p.stat().st_mtime)

if existing_csv and existing_npy and not force_recompute_embeddings:
    latest_csv = existing_csv[-1]
    latest_npy = existing_npy[-1]

    emb_df = pd.read_csv(latest_csv)
    emb_matrix = np.load(latest_npy)
    base_name = latest_csv.stem

    print("Partido:", partido_atual)
    print("Output dir:", output_dir)
    print("Reutilizando embeddings já existentes:")
    print("CSV:", latest_csv.name)
    print("NPY:", latest_npy.name)
else:
    emb_df, emb_matrix, base_name = generate_agenda_embeddings_from_txt(
        txt_path=str(txt_path),
        similarity_threshold=0.45,
        min_sentences_per_chunk=1,
        max_sentences_per_chunk=None,
        batch_size=32,
        output_dir=str(output_dir),
        save_files=True,
    )

    print("Partido:", partido_atual)
    print("Output dir:", output_dir)
    print("Base name:", base_name)

print("Chunks gerados:", len(emb_df))
print("Shape embeddings:", emb_matrix.shape)
display(emb_df[["source_id", "chunk_id", "chunk_text"]].head(5))

Partido: UNIAO
Output dir: /home/gpcmoura/Documents/Code/Msc/Repositories/political-discourse-analysis/data/party_agenda/embeddings/UNIAO
Reutilizando embeddings já existentes:
CSV: agenda_embeddings_Manifesto_Uniao_BRASIL.layout_aware.txt_20260411_1357.csv
NPY: agenda_embeddings_Manifesto_Uniao_BRASIL.layout_aware.txt_20260411_1357.npy
Chunks gerados: 105
Shape embeddings: (105, 384)


,source_id,chunk_id,chunk_text
0,Manifesto_Uniao_BRASIL.layout_aware.txt,0,Nossos Valores e Princípios O Brasil é reconhe...
1,Manifesto_Uniao_BRASIL.layout_aware.txt,1,"São muitas e diferentes as nossas origens, as ..."
2,Manifesto_Uniao_BRASIL.layout_aware.txt,2,É ele que nunca deixa de pulsar em nossos cora...
3,Manifesto_Uniao_BRASIL.layout_aware.txt,3,Esse Amor pelo Brasil está assentado na nossa ...
4,Manifesto_Uniao_BRASIL.layout_aware.txt,4,Mas não é uma utopia.


In [5]:
# Inspeção manual de alguns chunks
n_amostras = 3
for i, chunk in enumerate(emb_df["chunk_text"].head(n_amostras), start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk[:1200])  # limita visualização para não poluir a saída


--- Chunk 1 ---
Nossos Valores e Princípios O Brasil é reconhecido e celebrado por sua pluralidade.

--- Chunk 2 ---
São muitas e diferentes as nossas origens, as nossas culturas, as nossas tradições e costumes, os nossos valores, as nossas ideologias. Diversa é a nossa geografia, rica e variada é a nossa natureza. Somos um país continental, que abriga em seu vasto território todo tipo de gente, de pensamento, de profissão de fé, de posição política. Em meio a essas diferenças, uma coisa nos une e nos constitui como coletividade: o Amor por esse país.

--- Chunk 3 ---
É ele que nunca deixa de pulsar em nossos corações, que nos faz reagir ao pessimismo, que nos dá alento para seguirmos em frente, acreditando e lutando por um futuro melhor.


## 3) Tópicos da agenda (LDA)

Nesta etapa, usamos os CSVs gerados no pré-processamento em:
`data/party_agenda/preprocessing/tokenization/tokens/<PARTIDO>/CSV`

A modelagem de tópicos salva os artefatos em:
`data/party_agenda/topics/<PARTIDO>/`

Arquivos gerados incluem:
- `lda_model.model`;
- `lda_dictionary.dict`;
- `lda_topN_docs_por_topico.csv`;
- `lda_distribuicao_docs.csv`;
- `lda_topicos_termos.csv`;
- `lda_coherence_scores.csv` (quando houver busca por coerência).

In [6]:
# Define o partido para modelagem de tópicos
partido_topicos = elemento_teste if elemento_teste else "UNIAO"

# Carrega os CSVs de tokens da agenda
pasta_tokens_base = PROJECT_ROOT / "data" / "party_agenda" / "preprocessing" / "tokenization" / "tokens"
agenda_df = load_party_tokens_dataframe(
    pasta_tokens_base=pasta_tokens_base,
    partido=partido_topicos,
)

# Caminho de saída dos tópicos
topics_output_dir = PROJECT_ROOT / "data" / "party_agenda" / "topics" / partido_topicos

# Remove artefatos antigos para sobrescrever completamente
if topics_output_dir.exists():
    for path in topics_output_dir.glob("*"):
        if path.is_file():
            path.unlink()

# Executa modelagem de tópicos
topics_result = topics_main(
    dataframe=agenda_df,
    partido=partido_topicos,
    top_n=5,
    topic_start=2,
    topic_limit=8,
    topic_step=1,
    coherence_processes=1,
    search_passes=10,
    search_iterations=100,
    final_passes=30,
    final_iterations=300,
    output_base_dir=PROJECT_ROOT / "data" / "party_agenda" / "topics",
)

print("Partido:", partido_topicos)
print("Documentos usados:", len(agenda_df))
print("Número de tópicos selecionado:", topics_result["selected_topic_num"])

print("Saída:", topics_result["output_dir"])

print("\nTop termos por tópico:")
display(topics_result["topic_terms"])

print("\nTop documentos por tópico:")
display(topics_result["top_docs_per_topic_n"].head(10))

if not topics_result["coherence_scores"].empty:
    print("\nCoherence scores:")
    display(topics_result["coherence_scores"])

....................................
... Função topics_main iniciada! ...
... Partido: UNIAO ...
... Total de documentos considerados: 105 ...
... Identificando valor mais adequado de tópicos ...
...........................................
... Executando compute_coherence_values ...
... [1/6] Testando 2 tópicos ...
    Tópico 0: 0.065*"brasil" + 0.028*"capacidade" + 0.015*"união" + 0.015*"lutar" + 0.015*"bom" + 0.014*"criatividade" + 0.014*"mão" + 0.014*"escolher" + 0.014*"poder" + 0.008*"querer"
    Tópico 1: 0.011*"político" + 0.009*"brasileiro" + 0.006*"público" + 0.006*"brasil" + 0.006*"compromisso" + 0.005*"grande" + 0.005*"todo" + 0.005*"democracia" + 0.005*"país" + 0.004*"força"
... [1/6] concluído em 0.2s ...
... [2/6] Testando 3 tópicos ...
    Tópico 0: 0.079*"brasil" + 0.033*"capacidade" + 0.018*"união" + 0.018*"poder" + 0.017*"bom" + 0.017*"lutar" + 0.017*"escolher" + 0.017*"criatividade" + 0.010*"futuro" + 0.010*"brasileiro"
    Tópico 1: 0.013*"político" + 0.008*"público"

,topic,terms
0,0,"0.016*""brasil"" + 0.012*""brasileiro"" + 0.010*""p..."
1,1,"0.012*""político"" + 0.010*""público"" + 0.007*""de..."



Top documentos por tópico:


,doc_id,topic,probability,source_file,preprocess_agenda
0,17,0,0.999764,Manifesto_Uniao_BRASIL.layout_aware_preprocess...,avançar obra construção nacional lançar luz so...
1,92,0,0.99972,Manifesto_Uniao_BRASIL.layout_aware_preprocess...,sempre preservar integral fidelidade interesse...
2,50,0,0.999532,Manifesto_Uniao_BRASIL.layout_aware_preprocess...,16 17 18 priorização político público voltar p...
3,69,0,0.999453,Manifesto_Uniao_BRASIL.layout_aware_preprocess...,poder contar apenas resiliência ecossistema as...
4,78,0,0.999353,Manifesto_Uniao_BRASIL.layout_aware_preprocess...,acreditar construção estado eficiente fiel obr...
5,88,1,0.999536,Manifesto_Uniao_BRASIL.layout_aware_preprocess...,respeito cidadão excelência prestação serviço ...
6,39,1,0.999478,Manifesto_Uniao_BRASIL.layout_aware_preprocess...,manter porém clareza tratar solução insuficien...
7,90,1,0.999428,Manifesto_Uniao_BRASIL.layout_aware_preprocess...,triste dado revelar recente pesquisa nacional ...
8,65,1,0.999377,Manifesto_Uniao_BRASIL.layout_aware_preprocess...,último década agro tornar pujante segmento eco...
9,30,1,0.999227,Manifesto_Uniao_BRASIL.layout_aware_preprocess...,zelo patrimônio liberdade expressão condição f...



Coherence scores:


,num_topics,coherence_c_v
0,2,0.484183
1,3,0.391042
2,4,0.422541
3,5,0.442559
4,6,0.426013
5,7,0.427008


In [7]:
# Gera embeddings dos termos dos topicos da pauta (lda_topicos_termos.csv)
import json
import re
from sentence_transformers import SentenceTransformer

party_name = partido_topicos if "partido_topicos" in globals() else (elemento_teste if elemento_teste else "UNIAO")
topics_terms_path = PROJECT_ROOT / "data" / "party_agenda" / "topics" / party_name / "lda_topicos_termos.csv"
agenda_topics_df = pd.read_csv(topics_terms_path)

def extract_terms(text: str) -> str:
    terms = re.findall(r"\"([^\"]+)\"", str(text))
    if terms:
        return " ".join(terms)
    cleaned = re.sub(r"\d+\.\d+\*", " ", str(text))
    return re.sub(r"\s+", " ", cleaned.replace("+", " ")).strip()

agenda_topics_df["terms_clean"] = agenda_topics_df["terms"].apply(extract_terms)
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(model_name)
agenda_embeddings = model.encode(agenda_topics_df["terms_clean"].tolist(), normalize_embeddings=True)
agenda_topics_df["embedding"] = [json.dumps(vec.tolist(), ensure_ascii=False) for vec in agenda_embeddings]

agenda_embeddings_dir = PROJECT_ROOT / "data" / "party_agenda" / "embeddings" / party_name
agenda_embeddings_dir.mkdir(parents=True, exist_ok=True)
agenda_embeddings_path = agenda_embeddings_dir / "topicos_embeddings_termos.csv"
agenda_topics_df.to_csv(agenda_embeddings_path, index=False, encoding="utf-8")

print("Embeddings de topicos da pauta salvos em:", agenda_embeddings_path)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7667.83it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings de topicos da pauta salvos em: /home/gpcmoura/Documents/Code/Msc/Repositories/political-discourse-analysis/data/party_agenda/embeddings/UNIAO/topicos_embeddings_termos.csv


In [ ]:
# Calcula similaridade cosseno entre topicos da pauta e topicos dos discursos
import json  # carrega embeddings salvos
import numpy as np  # opera com matrizes
import pandas as pd  # manipula tabelas
from sentence_transformers import util  # calcula cosseno
party_name = partido_topicos if "partido_topicos" in globals() else (elemento_teste if elemento_teste else "UNIAO")  # define partido
agenda_topics_path = PROJECT_ROOT / "data" / "party_agenda" / "embeddings" / party_name / "topicos_embeddings_termos.csv"  # caminho pauta
discourse_topics_path = PROJECT_ROOT / "data" / "discourses" / "embeddings" / "discourses" / party_name / "topicos_embeddings_termos.csv"  # caminho discursos
agenda_topics_df = pd.read_csv(agenda_topics_path)  # le topicos da pauta
discourse_topics_df = pd.read_csv(discourse_topics_path)  # le topicos dos discursos
agenda_vecs = np.array([json.loads(v) for v in agenda_topics_df["embedding"]])  # vetoriza embeddings da pauta
discourse_vecs = np.array([json.loads(v) for v in discourse_topics_df["embedding"]])  # vetoriza embeddings dos discursos
sim_matrix = util.cos_sim(agenda_vecs, discourse_vecs).cpu().numpy()  # matriz de similaridade
best_idx = sim_matrix.argmax(axis=1)  # melhor topico de discurso por topico da pauta
best_score = sim_matrix.max(axis=1)  # melhor score por topico da pauta
result_rows = []  # prepara linhas do resultado
for i, j in enumerate(best_idx):  # percorre cada topico da pauta
    result_rows.append({"agenda_topic": int(agenda_topics_df.loc[i, "topic"]), "agenda_terms": agenda_topics_df.loc[i, "terms"], "discourse_topic": int(discourse_topics_df.loc[j, "topic"]), "discourse_terms": discourse_topics_df.loc[j, "terms"], "cosine_similarity": float(best_score[i])})  # adiciona linha
similarity_df = pd.DataFrame(result_rows)  # cria dataframe final
similarity_out = PROJECT_ROOT / "data" / "party_agenda" / "embeddings" / party_name / "topicos_similaridade_discursos.csv"  # saida csv
similarity_df.to_csv(similarity_out, index=False, encoding="utf-8")  # salva csv
print("Similaridade salva em:", similarity_out)  # mostra caminho